# CNN для классификации фруктов и овощей

Цирулев НВ М8О-408Б-22

## 1. Выбор начальных условий

Фиксируем задачу многоклассовой классификации фруктов и овощей, путь к датасету, устройство обучения и основные гиперпараметры экспериментов.

Подключаем библиотеки и локальные модули проекта, фиксируем параметры эксперимента и загружаем датасет через Kaggle API, если он еще не находится в `data/`.

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fruitveg_lab.data import create_dataloaders, describe_dataset, ensure_kaggle_dataset, make_transforms
from fruitveg_lab.models import ProduceResidualCNN, create_torchvision_resnet
from fruitveg_lab.plotting import plot_confusion_matrix, plot_history, show_image_grid
from fruitveg_lab.training import RunConfig, fit_classifier, get_device, make_conclusion, print_report, seed_everything, summarize_results

FAST_DEV_RUN = False
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2
BASE_EPOCHS = 6
SEED = 42

DATA_DIR = PROJECT_ROOT / "data" / "fruit-and-vegetable-image-recognition"
DATA_ROOT = ensure_kaggle_dataset(DATA_DIR)
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cnn"
DEVICE = get_device()

warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes should be converted to RGBA images",
    category=UserWarning,
    module="PIL.Image",
)

seed_everything(SEED)
print(f"project: {PROJECT_ROOT}")
print(f"dataset: {DATA_ROOT}")
print(f"device: {DEVICE}")

### 1.a. Набор данных и практическая задача

Датасет используется для распознавания продукта по фотографии. Практический смысл: кассы самообслуживания, складской учет, приложения питания и автоматическая маркировка товаров.

Проверяем структуру датасета и формируем таблицу количества изображений по классам в `train`, `validation` и `test`.

In [ ]:
info = describe_dataset(DATA_ROOT)
counts = pd.DataFrame(info.split_counts).fillna(0).astype(int)
counts["total"] = counts.sum(axis=1)
display(counts)
print(f"classes: {len(info.classes)}")
print(f"train images: {counts['train'].sum()}")
print(f"validation images: {counts['validation'].sum()}")
print(f"test images: {counts['test'].sum()}")

### 1.b. Метрики качества

Основные метрики: `accuracy`, macro/weighted `precision`, `recall`, `F1`, `top-3 accuracy`, classification report и confusion matrix. Для выбора лучшей конфигурации используется `validation macro F1`, потому что все классы важны одинаково.

Создаем DataLoader-объекты для выбранного режима преобразований и визуализируем примеры обучающих изображений.

In [ ]:
def loaders_for(transform_mode: str, batch_size: int = BATCH_SIZE):
    train_transform = make_transforms(IMAGE_SIZE, mode=transform_mode, normalize=True)
    eval_transform = make_transforms(IMAGE_SIZE, mode="plain", normalize=True)
    return create_dataloaders(
        DATA_ROOT,
        train_transform=train_transform,
        eval_transform=eval_transform,
        batch_size=batch_size,
        num_workers=NUM_WORKERS,
    )

preview_loaders = loaders_for("plain", batch_size=12)
show_image_grid(preview_loaders.train, preview_loaders.class_names, count=12, normalized=True)

## 2. Создание бейзлайна и оценка качества

Пункты 2.a-2.b: обучается сверточный baseline `ResNet-18` из `torchvision` с новой классификационной головой на 36 классов и оценивается качество по выбранным метрикам.

Обучаем baseline `torchvision ResNet-18`, строим графики обучения, classification report и confusion matrix на test split.

In [ ]:
num_classes = len(preview_loaders.class_names)
plain_loaders = loaders_for("plain")

cnn_baseline = fit_classifier(
    create_torchvision_resnet("resnet18", num_classes=num_classes, weights=None),
    name="torchvision_resnet18_plain",
    train_loader=plain_loaders.train,
    val_loader=plain_loaders.val,
    test_loader=plain_loaders.test,
    class_names=plain_loaders.class_names,
    config=RunConfig(
        epochs=BASE_EPOCHS,
        learning_rate=1e-3,
        weight_decay=1e-4,
        scheduler="none",
        fast_dev_run=FAST_DEV_RUN,
        seed=SEED,
    ),
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

plot_history(cnn_baseline)
print_report(cnn_baseline)
plot_confusion_matrix(cnn_baseline)

## 3. Улучшение бейзлайна

Пункты 3.a-3.g: формулируем и проверяем гипотезы про learning rate, аугментации, scheduler и более глубокую `ResNet-34`; лучшую конфигурацию выбираем по `validation macro F1` и сравниваем с baseline.

Проводим серию CNN-экспериментов, сохраняем результаты и формируем сравнительные таблицы для validation и test split.

In [ ]:
cnn_specs = [
    {
        "name": "resnet18_lr_3e-4",
        "model": "resnet18",
        "transform_mode": "plain",
        "learning_rate": 3e-4,
        "weight_decay": 1e-4,
        "scheduler": "none",
        "epochs": BASE_EPOCHS,
    },
    {
        "name": "resnet18_augmented",
        "model": "resnet18",
        "transform_mode": "augmented",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "scheduler": "none",
        "epochs": BASE_EPOCHS,
    },
    {
        "name": "resnet18_augmented_cosine",
        "model": "resnet18",
        "transform_mode": "augmented",
        "learning_rate": 5e-4,
        "weight_decay": 2e-4,
        "scheduler": "cosine",
        "epochs": BASE_EPOCHS + 2,
    },
    {
        "name": "resnet34_augmented_cosine",
        "model": "resnet34",
        "transform_mode": "augmented",
        "learning_rate": 5e-4,
        "weight_decay": 2e-4,
        "scheduler": "cosine",
        "epochs": BASE_EPOCHS + 2,
    },
]

cnn_experiment_records = []
for spec in cnn_specs:
    loaders = loaders_for(spec["transform_mode"])
    result = fit_classifier(
        create_torchvision_resnet(spec["model"], num_classes=num_classes, weights=None),
        name=spec["name"],
        train_loader=loaders.train,
        val_loader=loaders.val,
        test_loader=loaders.test,
        class_names=loaders.class_names,
        config=RunConfig(
            epochs=spec["epochs"],
            learning_rate=spec["learning_rate"],
            weight_decay=spec["weight_decay"],
            scheduler=spec["scheduler"],
            fast_dev_run=FAST_DEV_RUN,
            seed=SEED,
        ),
        device=DEVICE,
        output_dir=OUTPUT_DIR,
    )
    cnn_experiment_records.append({"spec": spec, "result": result})

cnn_hypothesis_results = [record["result"] for record in cnn_experiment_records]
display(summarize_results([cnn_baseline] + cnn_hypothesis_results, split="val"))
display(summarize_results([cnn_baseline] + cnn_hypothesis_results, split="test"))

### 3.c-3.g. Формирование и оценка улучшенного бейзлайна

После проверки гипотез выбираем лучшую конфигурацию, затем выводим ее подробные метрики и графики.

Выбираем лучший CNN baseline по `validation macro F1`, затем выводим его настройки, отчет классификации, историю обучения и confusion matrix.

In [ ]:
best_record = max(cnn_experiment_records, key=lambda item: item["result"]["val"]["macro_f1"])
best_cnn_spec = best_record["spec"]
best_torchvision_cnn = best_record["result"]

print("Best CNN spec:")
display(pd.DataFrame([best_cnn_spec]))
print_report(best_torchvision_cnn)
plot_history(best_torchvision_cnn)
plot_confusion_matrix(best_torchvision_cnn)

## 4. Имплементация алгоритма машинного обучения

Пункты 4.a-4.j: обучается собственная residual CNN без использования готовой модели `torchvision`, затем к ней применяются настройки улучшенного baseline и выполняется итоговое сравнение.

Обучаем собственную residual CNN в базовом режиме, после чего выводим метрики, графики и confusion matrix.

In [ ]:
custom_cnn_plain = fit_classifier(
    ProduceResidualCNN(num_classes=num_classes, dropout=0.25),
    name="custom_residual_cnn_plain",
    train_loader=plain_loaders.train,
    val_loader=plain_loaders.val,
    test_loader=plain_loaders.test,
    class_names=plain_loaders.class_names,
    config=RunConfig(
        epochs=BASE_EPOCHS,
        learning_rate=1e-3,
        weight_decay=1e-4,
        scheduler="none",
        fast_dev_run=FAST_DEV_RUN,
        seed=SEED,
    ),
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

plot_history(custom_cnn_plain)
print_report(custom_cnn_plain)
plot_confusion_matrix(custom_cnn_plain)

Обучаем собственную residual CNN с настройками, выбранными на этапе улучшения baseline.

In [ ]:
improved_loaders = loaders_for(best_cnn_spec["transform_mode"])
custom_cnn_improved = fit_classifier(
    ProduceResidualCNN(num_classes=num_classes, dropout=0.30),
    name="custom_residual_cnn_improved",
    train_loader=improved_loaders.train,
    val_loader=improved_loaders.val,
    test_loader=improved_loaders.test,
    class_names=improved_loaders.class_names,
    config=RunConfig(
        epochs=best_cnn_spec["epochs"],
        learning_rate=best_cnn_spec["learning_rate"],
        weight_decay=best_cnn_spec["weight_decay"],
        scheduler=best_cnn_spec["scheduler"],
        fast_dev_run=FAST_DEV_RUN,
        seed=SEED,
    ),
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

plot_history(custom_cnn_improved)
print_report(custom_cnn_improved)
plot_confusion_matrix(custom_cnn_improved)

### 4.d-4.j. Сравнение и выводы

Сравниваются baseline, улучшенный `torchvision` baseline, собственная модель и собственная модель с улучшениями.

Формируем финальную таблицу результатов CNN-моделей на test split и краткий вывод по значениям метрик.

In [ ]:
cnn_final_results = [
    cnn_baseline,
    best_torchvision_cnn,
    custom_cnn_plain,
    custom_cnn_improved,
]

display(summarize_results(cnn_final_results, split="test"))
print(make_conclusion(cnn_final_results, split="test"))